In [1]:
# Setup
import math_toolkit
math_toolkit.activate()

# Convolution with a moving Fejer kernel

A convolution on the circle averages a periodic function near each point. The averaging window wraps around the endpoints, so points near $-\frac12$ can borrow information from points near $\frac12$.

The Fejer kernel is an approximate identity: as $N$ grows, the kernel gets taller and narrower, with most of its mass inside a region of scale about $1/N$.

This worksheet compares a function $f$ with its circular convolution

$$
(F_N*f)(x)=\int_{-1/2}^{1/2} F_N(x-t) f(t)\,dt.
$$

The slider $N$ changes the scale. The slider $x_0$ moves one copy of the kernel so you can see which part of the function is being averaged.

## The averaging kernel

On $S^1=\mathbb{R}/\mathbb{Z}$, the Fejer kernel is

$$
F_N(u)=\frac{1}{N}\left(\frac{\sin(\pi N u)}{\sin(\pi u)}\right)^2.
$$

It has total integral $1$ over one period. For drawing on the same axes as the functions, we plot the normalized shape $F_N(x-x_0)/N$.

In [2]:
# The removable singularity has value N at every integer u.
t = Symbol("t", real=True)
u = Symbol("u", real=True)
n = Symbol("n", integer=True, positive=True)
N = Symbol("N", integer=True, positive=True)
x_0 = Symbol("x_0", real=True)

Fejer = Function("Fejer", latex=r"F_N")(u, N) @EqDef@ (
    If(sin(pi * u) @Eq@ 0).Then(N)
    .Otherwise(sin(pi * N * u) ** 2 / (N * sin(pi * u) ** 2))
)

r"""
The kernel we move through the figures is
$$
F_N(x-x_0)={{ Fejer(x-x_0,N) }}.
$$
""" >> md


The kernel we move through the figures is
```{math}
:enumerated: false
F_N(x-x_0)=\begin{cases} N & \text{for}\: \sin{\left(\pi \left(x - x_{0}\right) \right)} = 0 \\\frac{\sin^{2}{\left(\pi N \left(x - x_{0}\right) \right)}}{N \sin^{2}{\left(\pi \left(x - x_{0}\right) \right)}} & \text{otherwise} \end{cases}.
```


## Square wave

The square wave is the sharp test case. Away from the jump, the convolution should move toward the original function. At the jump, every symmetric approximate identity sees both sides.

In [3]:
# Centered one-period representative of the square wave.
Sq = Function("Sq")(x) @EqDef@ (
    If(x > 0).Then(1)
    .If(x < 0).Then(-1)
    .Otherwise(0)
)

# Fejer convolution is the Cesaro mean of the Fourier series.
Sq_fejer = Function("Sq_fejer", latex=r"F_N*\mathrm{Sq}")(x, N) @EqDef@ Sum(
    (1 - n / N) * (2 * (1 - cos(pi * n)) / (pi * n)) * sin(2 * pi * n * x),
    (n, 1, N - 1),
)

r"""
On $-\frac{1}{2}\le x\le \frac{1}{2}$ the target is
$$
\mathrm{Sq}(x):={{ Sq(x)>>Expand }}.
$$

Its Fejer convolution is the weighted Fourier sum
$$
(F_N*\mathrm{Sq})(x)={{ Sq_fejer(x,N) }}.
$$
""" >> md


On $-\frac{1}{2}\le x\le \frac{1}{2}$ the target is
```{math}
:enumerated: false
\mathrm{Sq}(x):=\begin{cases} 1 & \text{for}\: x > 0 \\-1 & \text{for}\: x < 0 \\0 & \text{otherwise} \end{cases}.
```

Its Fejer convolution is the weighted Fourier sum
```{math}
:enumerated: false
(F_N*\mathrm{Sq})(x)=\sum_{n=1}^{N - 1} \frac{\left(1 - \frac{n}{N}\right) \left(2 - 2 \left(-1\right)^{n}\right) \sin{\left(2 \pi n x \right)}}{\pi n}.
```


In [7]:
fig1 = figure("Square wave Fejer convolution", new=True)
fig1.view.home_range = {"x": (-1/2, 1/2), "y": (-1.5, 1.5)}
fig1.show()

fig1.parameters({
    N: {"value": 8, "min": 2, "max": 80, "step": 1},
    x_0: {"value": 0, "min": -0.5, "max": 0.5, "step": 0.01},
})

fig1.plot(
    Sq(x),
    x,
    name="square wave",
    label=r"$\mathrm{Sq}(x)$",
    style={"width": 4, "opacity": 0.45},
    samples=1001,
)
fig1.plot(
    Sq_fejer(x, N),
    x,
    name="Fejer convolution",
    label=r"$(F_N*\mathrm{Sq})(x)$",
    style={"width": 2, "color": "#d62728"},
    samples=2001,
)
fig1.plot(
    Fejer(x - x_0, N) / N,
    x,
    name="moving kernel",
    label=r"$F_N(x-x_0)/N$",
    style={"width": 2, "color": "#2ca02c", "opacity": 0.85},
    samples=2001,
)

<lambdifygenerated-25>:5: RuntimeWarning: invalid value encountered in divide
  return x0*select([equal(x2, 0),True], [N,x0*sin(N*x1)**2/x2**2], default=nan)
<lambdifygenerated-26>:5: RuntimeWarning: invalid value encountered in divide
  return x0*select([equal(x2, 0),True], [N,x0*sin(N*x1)**2/x2**2], default=nan)


:::{admonition} Question
Move $x_0$ to a point far from the jump, then increase $N$. What part of the square wave is the kernel averaging?
:::

:::{admonition} Question
Move $x_0$ to the jump at $0$. Why does the convolution move toward $0$ there instead of toward $1$ or $-1$?
:::

## A noisy trigonometric signal

A function with small high-frequency terms behaves differently from the square wave. There are no jumps to split the average, but the narrow wiggles are easy for the Fejer kernel to damp.

In [8]:
# A low-frequency signal with several smaller high-frequency terms.
Noisy = Function("Noisy", latex=r"q")(x) @EqDef@ (
    0.55 * sin(2 * pi * x)
    + 0.25 * cos(4 * pi * x)
    - 0.20 * sin(6 * pi * x)
    + 0.12 * cos(20 * pi * x)
    - 0.08 * sin(34 * pi * x)
    + 0.06 * cos(58 * pi * x)
)

Noisy_fejer = Function("Noisy_fejer", latex=r"F_N*q")(x, N) @EqDef@ (
    If(N > 1).Then((1 - 1 / N) * 0.55 * sin(2 * pi * x)).Otherwise(0)
    + If(N > 2).Then((1 - 2 / N) * 0.25 * cos(4 * pi * x)).Otherwise(0)
    - If(N > 3).Then((1 - 3 / N) * 0.20 * sin(6 * pi * x)).Otherwise(0)
    + If(N > 10).Then((1 - 10 / N) * 0.12 * cos(20 * pi * x)).Otherwise(0)
    - If(N > 17).Then((1 - 17 / N) * 0.08 * sin(34 * pi * x)).Otherwise(0)
    + If(N > 29).Then((1 - 29 / N) * 0.06 * cos(58 * pi * x)).Otherwise(0)
)

r"""
The noisy function is
$$
q(x)={{ Noisy(x) }}.
$$

Each Fourier mode is multiplied by the Fejer weight $1-k/N$ when $k<N$, and is removed when $k\ge N$.
""" >> md


The noisy function is
```{math}
:enumerated: false
q(x)=0.55 \sin{\left(2 \pi x \right)} - 0.2 \sin{\left(6 \pi x \right)} - 0.08 \sin{\left(34 \pi x \right)} + 0.25 \cos{\left(4 \pi x \right)} + 0.12 \cos{\left(20 \pi x \right)} + 0.06 \cos{\left(58 \pi x \right)}.
```

Each Fourier mode is multiplied by the Fejer weight $1-k/N$ when $k<N$, and is removed when $k\ge N$.


In [10]:
fig2 = figure("Noisy function Fejer convolution", new=True)
fig2.view.home_range = {"x": (-1/2, 1/2), "y": (-1.6, 1.6)}
fig2.show()

fig2.parameters({
    N: {"value": 8, "min": 2, "max": 80, "step": 1},
    x_0: {"value": 0, "min": -0.5, "max": 0.5, "step": 0.01},
})

fig2.plot(
    Noisy(x),
    x,
    name="noisy function",
    label=r"$q(x)$",
    style={"width": 4, "opacity": 0.45},
    samples=2001,
)
fig2.plot(
    Noisy_fejer(x, N),
    x,
    name="Fejer convolution",
    label=r"$(F_N*q)(x)$",
    style={"width": 2, "color": "#d62728"},
    samples=2001,
)
fig2.plot(
    Fejer(x - x_0, N) / N,
    x,
    name="moving kernel",
    label=r"$F_N(x-x_0)/N$",
    style={"width": 2, "color": "#2ca02c", "opacity": 0.85},
    samples=2001,
)

<lambdifygenerated-37>:5: RuntimeWarning: invalid value encountered in divide
  return x0*select([equal(x2, 0),True], [N,x0*sin(N*x1)**2/x2**2], default=nan)
<lambdifygenerated-38>:5: RuntimeWarning: invalid value encountered in divide
  return x0*select([equal(x2, 0),True], [N,x0*sin(N*x1)**2/x2**2], default=nan)


:::{admonition} Question
Start with small $N$. Which wiggles disappear first? Increase $N$ and watch when the modes with frequencies $10$, $17$, and $29$ return.
:::

:::{admonition} Question
For the square wave, large $N$ still leaves a visible overshoot near the jump. For $q(x)$, what changes as soon as the kernel becomes narrow enough to see the fast oscillations?
:::